<a href="https://colab.research.google.com/github/knowledgegirlee-ux/Hospital_Readmissions/blob/main/churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

---

## 1.Introduction & Market Analysis


Rather than predicting *who* churns, we quantify *how much revenue is at risk*
and *where intervention pays off*. We combine a churn probability with each
customer's lifetime value to produce a Revenue-at-Risk score, then concentrate
retention spend on the highest-ROI segment. The recommendation: don't spend
equally on retention — spend where it counts.

---

## 2.Setup and Data Loading


### 2.1 Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV


from xgboost import XGBClassifier, plot_importance

pd.set_option("display.max_columns", 120)


### 2.2 Connect to Google Drive

If you are running this notebook on Google Colab, mount your Drive and `cd` into the folder that contains the dataset.


In [ ]:
# Run this cell only on Google Colab.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#%cd "/content/drive/MyDrive/WhereThisNotebookIsLocated"
%cd "/content/drive/MyDrive/finalassginment99"

In [ ]:
import os
from pathlib import Path

current_dir = Path(os.getcwd())
client_path = current_dir / "telecom" / "Client.csv"
record_path = current_dir / "telecom" / "Record.csv"

for p in (client_path, record_path):
    print(f"{'OK ' if p.exists() else 'MISSING '} {p}")

In [ ]:
client = pd.read_csv(client_path)
record = pd.read_csv(record_path)

print(f"Client: {client.shape}")
print(f"Record: {record.shape}")

In [ ]:
# Merge on Customer_ID. Both tables have one row per customer, so this is a 1:1 join.
df = record.merge(client, on='Customer_ID', how='inner')
print(f"Merged: {df.shape}")
df.head()

In [ ]:
df.info()
df.info(verbose=False)


In [ ]:
# Count of missing values per column
missing_counts = df.isna().sum()

# Take the 20 columns with the most missing values
top_missing = missing_counts.sort_values(ascending=False).head(20)

# Plot
sns.barplot(x=top_missing.values, y=top_missing.index)
plt.xlabel('Number of missing values')
plt.title('Top 20 columns by number of missing values')
plt.show()

In [ ]:
# Counts and proportions
print(df['churn'].value_counts())
print()
print(df['churn'].value_counts(normalize=True))

# Bar chart
counts = df['churn'].value_counts().sort_index()
labels = ['stayed (0)', 'churned (1)']

plt.figure(figsize=(5, 4))
plt.bar(labels, counts.values)
plt.title('Churn distribution')
plt.ylabel('Number of customers')
plt.show()

Churn is near 50%

In [ ]:
# Split eqpdays into two groups based on churn
stayed = df[df['churn'] == 0]['eqpdays']
churned = df[df['churn'] == 1]['eqpdays']

# Compare the averages
print(f'Mean equipment age (stayed):  {stayed.mean():.1f} days')
print(f'Mean equipment age (churned): {churned.mean():.1f} days')

# Overlay the two distributions
plt.figure(figsize=(8, 5))
plt.hist(stayed, bins=30, alpha=0.5, label='stayed')
plt.hist(churned, bins=30, alpha=0.5, label='churned')
plt.xlabel('Equipment age (days)')
plt.ylabel('Number of customers')
plt.title('Equipment age: stayers vs churners')
plt.legend()
plt.show()

In [ ]:
df.groupby('churn')['custcare_Mean'].mean().plot(kind='bar')
plt.title('Avg Customer Care Calls: Stayed vs Churned')
plt.ylabel('Mean Monthly Support Calls')
plt.xticks([0,1], ['Stayed', 'Churned'], rotation=0)
plt.show()

# Usage change: stayers vs churners
df.groupby('churn')['change_mou'].mean().plot(kind='bar')
plt.title('Avg Change in Usage: Stayed vs Churned')
plt.ylabel('% Change in Minutes of Use')
plt.xticks([0,1], ['Stayed', 'Churned'], rotation=0)
plt.show()


In [ ]:
# Pairplot with log-transformed skewed variables
plot_df = df[[
    'months',
    'rev_Mean',
    'mou_Mean',
    'totmrc_Mean',
    'eqpdays',
    'churn'
]].dropna().copy()

# Log-transform highly skewed positive variables
plot_df['log_rev_Mean'] = np.log1p(plot_df['rev_Mean'])
plot_df['log_mou_Mean'] = np.log1p(plot_df['mou_Mean'])
plot_df['log_totmrc_Mean'] = np.log1p(plot_df['totmrc_Mean'])
plot_df['log_eqpdays'] = np.log1p(plot_df['eqpdays'])

plot_df['churn_label'] = plot_df['churn'].map({
    0: 'stayed',
    1: 'churned'
})

pair_cols = [
    'months',
    'log_rev_Mean',
    'log_mou_Mean',
    'log_totmrc_Mean',
    'log_eqpdays',
    'churn_label'
]

pair_sample = plot_df[pair_cols].sample(n=5000, random_state=42)

sns.pairplot(
    pair_sample,
    hue='churn_label',
    corner=True,
    diag_kind='kde',
)

plt.suptitle('Pairwise view of tenure, revenue, usage, equipment age, and churn', y=1.02)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score # Ensure this is imported for model evaluation steps

# --- Replicate df_fe creation (from eWgszeUvPu9L) ---
df_fe = df.copy()
eps = 1e-6
df_fe['usage_decay_3m']   = df_fe['avg3mou'] / (df_fe['avg6mou'] + eps)
df_fe['revenue_decay_3m'] = df_fe['avg3rev'] / (df_fe['avg6rev'] + eps)
df_fe['mou_change_ratio'] = df_fe['change_mou'] / (df_fe['mou_Mean'] + eps)
df_fe['eqp_per_tenure'] = df_fe['eqpdays'] / (df_fe['months'] * 30 + eps)
df_fe['eqp_age_years']  = df_fe['eqpdays'] / 365.0
df_fe['failed_call_rate'] = (df_fe['drop_vce_Mean'] + df_fe['blck_vce_Mean'] +
                             df_fe['unan_vce_Mean']) / (df_fe['plcd_vce_Mean'] + eps)
df_fe['drop_blk_rate']    = df_fe['drop_blk_Mean'] / (df_fe['attempt_Mean'] + eps)
df_fe['completion_rate']  = df_fe['complete_Mean'] / (df_fe['attempt_Mean'] + eps)
df_fe['care_intensity'] = df_fe['custcare_Mean'] / (df_fe['mou_Mean'] + eps)
df_fe['overage_share']   = df_fe['ovrrev_Mean'] / (df_fe['rev_Mean'] + eps)
df_fe['revenue_per_min'] = df_fe['rev_Mean'] / (df_fe['mou_Mean'] + eps)
df_fe['subs_inactive_ratio'] = (df_fe['uniqsubs'] - df_fe['actvsubs']) / (df_fe['uniqsubs'] + eps)
df_fe['revenue_per_month']   = df_fe['totrev'] / (df_fe['months'] + eps)
df_fe = df_fe.replace([np.inf, -np.inf], np.nan)

# --- Replicate df_clean, X, y, X_train, y_train, train_medians, scaler creation (from bafc4ed8) ---
df_clean = df_fe.drop(columns=['Customer_ID'])
missing_rate = df_clean.isna().mean()
cols_to_drop = missing_rate[missing_rate > 0.40].index.tolist()
df_clean = df_clean.drop(columns=cols_to_drop)
object_cols = df_clean.select_dtypes(include='object').columns.tolist()
for col in object_cols:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col].astype(str))
X = df_clean.drop('churn', axis=1)
y = df_clean['churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)
scaler = StandardScaler().fit(X_train)
# Xtr_s is needed for calibrated classifier fit if scaling is used, even if not directly for XGBoost predict
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

# --- Replicate best model identification (from 4968c5bd, XPgvEOKaQtE1) ---
best = XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
best.fit(X_train, y_train) # Fit the best model
needs_scale = False # For XGBoost, needs_scale is False

# --- Replicate df_scored creation (from xbGR0U1bTAtE) ---
X_all = X.fillna(train_medians)
X_all_in = scaler.transform(X_all) if needs_scale else X_all # Ensure X_all_in is correctly prepared
final_clf = CalibratedClassifierCV(best, method='isotonic', cv=3).fit(
    (Xtr_s if needs_scale else X_train), y_train
)

df_scored = df_clean.copy()
df_scored['churn_proba'] = final_clf.predict_proba(X_all_in)[:, 1]

MARGIN       = 0.45
HORIZON_M    = 36
MONTHLY_DISC = 0.008
pv_factor = sum(1 / (1 + MONTHLY_DISC) ** m for m in range(1, HORIZON_M + 1))

monthly_rev = df_scored['rev_Mean'].clip(lower=0).fillna(df_scored['rev_Mean'].median())
df_scored['CLV'] = monthly_rev * MARGIN * pv_factor
df_scored['revenue_at_risk'] = df_scored['churn_proba'] * df_scored['CLV']


plt.figure(figsize=(10, 6))

# Generate the scatterplot
sns.scatterplot(
    data=df_scored,               # Data source
    x='rev_Mean',               # X-axis
    y='churn_proba',            # Y-axis
    hue='revenue_at_risk',      # Color points by this column
    palette='viridis',          # Choose a colormap (e.g., 'viridis', 'coolwarm')
    alpha=0.7,                  # Adjust transparency for overlapping points
    edgecolor=None
)

plt.title('Churn Probability vs. Mean Revenue (Colored by Revenue at Risk)')
plt.xlabel('Mean Revenue (rev_Mean)')
plt.ylabel('Churn Probability (churn_proba)')

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Re-defining plot_df to ensure it's available in this cell
plot_df = df[[
    'months',
    'rev_Mean',
    'mou_Mean',
    'totmrc_Mean',
    'eqpdays',
    'churn'
]].dropna().copy()

plt.figure(figsize=(10, 6))
sns.histplot(plot_df['rev_Mean'], kde=True, bins=50)
plt.title('Distribution of Average Monthly Revenue (rev_Mean)')
plt.xlabel('Average Monthly Revenue')
plt.ylabel('Count')
plt.show()

The pairplot does not show a clean geometric separation between churners and non-churners.
This suggests that churn is not explained by a single simple numeric variable such as revenue or usage alone.
However, the diagonal distributions indicate that equipment age (`eqpdays`) is somewhat shifted for churned customers, making it a more interpretable candidate for business intervention.
Revenue- and usage-related variables appear correlated with each other, so they may act as overlapping proxies rather than independent explanations.

I will reframe churn from *"who leaves?"* to **"how much revenue is at risk, and
where is intervention worth it?"**

- **Target & ML task:** the binary `churn` flag — a **classification** task.
- **Deliverable:** a **Revenue-at-Risk score** = calibrated P(churn) × Customer
  Lifetime Value (CLV), used to tier customers into a value × risk matrix.
- **Business question (one sentence):** *Which customers should we spend
  retention budget on to protect the most lifetime revenue per dollar spent?*
- **Who acts:** the retention / CRM team, using each customer's segment.
- **Recommendation:** concentrate retention spend on the *High-Value /
  High-Risk* quadrant, where ROI is highest — not spread evenly across the base.


In [ ]:
df_fe = df.copy()
eps = 1e-6  # avoid divide-by-zero

# --- Usage decay & revenue trend (decline precedes churn) ---
df_fe['usage_decay_3m']   = df_fe['avg3mou'] / (df_fe['avg6mou'] + eps)
df_fe['revenue_decay_3m'] = df_fe['avg3rev'] / (df_fe['avg6rev'] + eps)
df_fe['mou_change_ratio'] = df_fe['change_mou'] / (df_fe['mou_Mean'] + eps)

# --- Equipment age relative to tenure (old phone + long customer = upgrade-due) ---
df_fe['eqp_per_tenure'] = df_fe['eqpdays'] / (df_fe['months'] * 30 + eps)
df_fe['eqp_age_years']  = df_fe['eqpdays'] / 365.0


# --- Service-quality friction (failed calls as share of attempts) ---
df_fe['failed_call_rate'] = (df_fe['drop_vce_Mean'] + df_fe['blck_vce_Mean'] +
                             df_fe['unan_vce_Mean']) / (df_fe['plcd_vce_Mean'] + eps)
df_fe['drop_blk_rate']    = df_fe['drop_blk_Mean'] / (df_fe['attempt_Mean'] + eps)
df_fe['completion_rate']  = df_fe['complete_Mean'] / (df_fe['attempt_Mean'] + eps)

# --- Support burden (frustration signal) ---
df_fe['care_intensity'] = df_fe['custcare_Mean'] / (df_fe['mou_Mean'] + eps)

# --- Revenue mix / overage dependence (bill-shock churn driver) ---
df_fe['overage_share']   = df_fe['ovrrev_Mean'] / (df_fe['rev_Mean'] + eps)
df_fe['revenue_per_min'] = df_fe['rev_Mean'] / (df_fe['mou_Mean'] + eps)

# --- Account complexity ---
df_fe['subs_inactive_ratio'] = (df_fe['uniqsubs'] - df_fe['actvsubs']) / (df_fe['uniqsubs'] + eps)
df_fe['revenue_per_month']   = df_fe['totrev'] / (df_fe['months'] + eps)

# Clean infinities from any residual division
df_fe = df_fe.replace([np.inf, -np.inf], np.nan)
print("Engineered features added. New shape:", df_fe.shape)


In [ ]:
df_clean = df_fe.drop(columns=['Customer_ID'])

# Drop columns >40% missing
missing_rate = df_clean.isna().mean()
cols_to_drop = missing_rate[missing_rate > 0.40].index.tolist()
df_clean = df_clean.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} high-missing cols:", cols_to_drop)

# Encode categoricals; NaN becomes its own level ('nan')
object_cols = df_clean.select_dtypes(include='object').columns.tolist()
for col in object_cols:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col].astype(str))

X = df_clean.drop('churn', axis=1)
y = df_clean['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Impute AFTER split (fit on train only) to avoid leakage
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)
print("Train/test ready:", X_train.shape, X_test.shape)


In [ ]:
!pip -q install lightgbm catboost
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score # This was missing too and will be needed.

# Logistic Regression needs scaling; trees do not
scaler = StandardScaler().fit(X_train)
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

models = {
    'LogisticRegression': (LogisticRegression(max_iter=1000, C=0.5), True),
    'RandomForest':       (RandomForestClassifier(n_estimators=400, max_depth=14,
                              n_jobs=-1, random_state=42), False),
    'XGBoost':            (XGBClassifier(n_estimators=500, learning_rate=0.03,
                              max_depth=6, subsample=0.8, colsample_bytree=0.8,
                              eval_metric='logloss', random_state=42), False),
    'LightGBM':           (LGBMClassifier(n_estimators=600, learning_rate=0.03,
                              num_leaves=48, subsample=0.8, random_state=42, verbose=-1), False),
    'CatBoost':           (CatBoostClassifier(iterations=600, learning_rate=0.03,
                              depth=6, verbose=0, random_state=42), False),
}

results = {}
for name, (mdl, needs_scale) in models.items():
    Xtr, Xte = (Xtr_s, Xte_s) if needs_scale else (X_train, X_test)
    mdl.fit(Xtr, y_train)
    proba = mdl.predict_proba(Xte)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    results[name] = {'model': mdl, 'needs_scale': needs_scale,
                     'auc': roc_auc_score(y_test, proba),
                     'acc': accuracy_score(y_test, pred)}
    print(f"{name:20s}  AUC={results[name]['auc']:.4f}  Acc={results[name]['acc']:.4f}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\nBest model: {best_name}  (AUC={results[best_name]['auc']:.4f})")

In [ ]:
best = results[best_name]['model']
needs_scale = results[best_name]['needs_scale']
Xtr, Xte = (Xtr_s, Xte_s) if needs_scale else (X_train, X_test)

# Isotonic calibration — makes P(churn) a true probability for CLV math
calibrated = CalibratedClassifierCV(best, method='isotonic', cv=3)
calibrated.fit(Xtr, y_train)
proba_cal = calibrated.predict_proba(Xte)[:, 1]
print(f"Calibrated AUC: {roc_auc_score(y_test, proba_cal):.4f}")


In [ ]:
cm = confusion_matrix(y_test, (proba_cal >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['stay','churn'],
            yticklabels=['stay','churn'], cmap='Blues')
plt.title(f'Confusion Matrix — {best_name} (calibrated)')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

# Re-define df_scored and its dependencies in case kernel state was lost or cells executed out of order.
# This assumes 'df' is loaded from earlier steps (cell 4e6df57b) and is available globally.

# Mimic df_fe creation (from eWgszeUvPu9L)
df_fe = df.copy()
eps = 1e-6
df_fe['usage_decay_3m']   = df_fe['avg3mou'] / (df_fe['avg6mou'] + eps)
df_fe['revenue_decay_3m'] = df_fe['avg3rev'] / (df_fe['avg6rev'] + eps)
df_fe['mou_change_ratio'] = df_fe['change_mou'] / (df_fe['mou_Mean'] + eps)
df_fe['eqp_per_tenure'] = df_fe['eqpdays'] / (df_fe['months'] * 30 + eps)
df_fe['eqp_age_years']  = df_fe['eqpdays'] / 365.0
df_fe['failed_call_rate'] = (df_fe['drop_vce_Mean'] + df_fe['blck_vce_Mean'] +
                             df_fe['unan_vce_Mean']) / (df_fe['plcd_vce_Mean'] + eps)
df_fe['drop_blk_rate']    = df_fe['drop_blk_Mean'] / (df_fe['attempt_Mean'] + eps)
df_fe['completion_rate']  = df_fe['complete_Mean'] / (df_fe['attempt_Mean'] + eps)
df_fe['care_intensity'] = df_fe['custcare_Mean'] / (df_fe['mou_Mean'] + eps)
df_fe['overage_share']   = df_fe['ovrrev_Mean'] / (df_fe['rev_Mean'] + eps)
df_fe['revenue_per_min'] = df_fe['rev_Mean'] / (df_fe['mou_Mean'] + eps)
df_fe['subs_inactive_ratio'] = (df_fe['uniqsubs'] - df_fe['actvsubs']) / (df_fe['uniqsubs'] + eps)
df_fe['revenue_per_month']   = df_fe['totrev'] / (df_fe['months'] + eps)
df_fe = df_fe.replace([np.inf, -np.inf], np.nan)

# Mimic df_clean, X, y, X_train, y_train, train_medians, scaler creation (from bafc4ed8)
df_clean = df_fe.drop(columns=['Customer_ID'])
missing_rate = df_clean.isna().mean()
cols_to_drop = missing_rate[missing_rate > 0.40].index.tolist()
df_clean = df_clean.drop(columns=cols_to_drop)
object_cols = df_clean.select_dtypes(include='object').columns.tolist()
for col in object_cols:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col].astype(str))
X = df_clean.drop('churn', axis=1)
y = df_clean['churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)
scaler = StandardScaler().fit(X_train)
# Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test) # Not strictly needed for final_clf with XGBoost

# Mimic best model identification (from 4968c5bd)
best = XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
best.fit(X_train, y_train) # Fit the best model

needs_scale = False # For XGBoost, needs_scale is False

# Re-create df_scored (from xbGR0U1bTAtE)
X_all = X.fillna(train_medians)
X_all_in = X_all # because needs_scale is False for XGBoost
final_clf = CalibratedClassifierCV(best, method='isotonic', cv=3).fit(X_train, y_train)

df_scored = df_clean.copy()
df_scored['churn_proba'] = final_clf.predict_proba(X_all_in)[:, 1]

MARGIN       = 0.45
HORIZON_M    = 36
MONTHLY_DISC = 0.008
pv_factor = sum(1 / (1 + MONTHLY_DISC) ** m for m in range(1, HORIZON_M + 1))

monthly_rev = df_scored['rev_Mean'].clip(lower=0).fillna(df_scored['rev_Mean'].median())
df_scored['CLV'] = monthly_rev * MARGIN * pv_factor
df_scored['revenue_at_risk'] = df_scored['churn_proba'] * df_scored['CLV']

# Original scatter plot code
sample = df_scored.sample(5000, random_state=42)
plt.figure(figsize=(8,5))
sc = plt.scatter(sample['churn_proba'], sample['rev_Mean'].clip(upper=200),
    c=sample['revenue_at_risk'], cmap='YlOrRd', alpha=0.4, s=8)
plt.axvline(x=0.5, color='gray', linestyle='--')
plt.axhline(y=df_scored['rev_Mean'].quantile(0.6), color='gray', linestyle='--')
plt.xlabel('Churn Probability'); plt.ylabel('Monthly Revenue ($)')
plt.title('Revenue-at-Risk Segmentation Scatter')
plt.colorbar(sc, label='Revenue-at-Risk Score')
plt.savefig('scatter.png', dpi=150, bbox_inches='tight')

In [ ]:
comp = pd.DataFrame(results).T[['auc','acc']].sort_values('auc', ascending=False)
comp.plot(kind='bar', figsize=(8,4))
plt.title('Model Comparison (AUC vs Accuracy)')
plt.ylabel('Score'); plt.ylim(0.5, 1.0); plt.xticks(rotation=30, ha='right')
plt.legend(['AUC','Accuracy']); plt.tight_layout(); plt.show()
print(comp.round(4))

In [ ]:
if hasattr(best, 'feature_importances_'):
    imp = pd.Series(best.feature_importances_, index=X_train.columns)
else:  # LogisticRegression
    imp = pd.Series(np.abs(best.coef_[0]), index=X_train.columns)
imp.sort_values(ascending=False).head(15).iloc[::-1].plot(kind='barh', figsize=(8,6))
plt.title(f'Top 15 Features — {best_name}')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()


In [ ]:
# Score all labelled customers with the calibrated model
X_all = X.fillna(train_medians)
X_all_in = scaler.transform(X_all) if needs_scale else X_all
final_clf = CalibratedClassifierCV(best, method='isotonic', cv=3).fit(
    (Xtr_s if needs_scale else X_train), y_train)

df_scored = df_clean.copy()
df_scored['churn_proba'] = final_clf.predict_proba(X_all_in)[:, 1]

# --- CLV assumptions (state these on your slides) ---
MARGIN       = 0.45     # gross margin on telecom revenue
HORIZON_M    = 36       # months
MONTHLY_DISC = 0.008    # ~10% annual discount

# Present-value annuity factor over the 36-month horizon
pv_factor = sum(1 / (1 + MONTHLY_DISC) ** m for m in range(1, HORIZON_M + 1))
print(f"PV annuity factor (36mo): {pv_factor:.2f}")

monthly_rev = df_scored['rev_Mean'].clip(lower=0).fillna(df_scored['rev_Mean'].median())
df_scored['CLV'] = monthly_rev * MARGIN * pv_factor

# Revenue-at-Risk = probability of leaving × value of the customer
df_scored['revenue_at_risk'] = df_scored['churn_proba'] * df_scored['CLV']

print(df_scored[['churn_proba','CLV','revenue_at_risk']].describe().round(2))
print(f"\nTotal portfolio CLV:   ${df_scored['CLV'].sum():,.0f}")
print(f"Total Revenue-at-Risk: ${df_scored['revenue_at_risk'].sum():,.0f}")


In [ ]:
value_threshold = df_scored['CLV'].quantile(0.60)   # top 40% = high value
RISK_THRESHOLD  = 0.50

def assign_segment(r):
    high_value = r['CLV'] >= value_threshold
    high_risk  = r['churn_proba'] >= RISK_THRESHOLD
    if   high_value and high_risk:     return 'Priority Retention'
    elif high_value and not high_risk: return 'Protect & Grow'
    elif not high_value and high_risk: return 'Standard Intervention'
    else:                              return 'Monitor'

df_scored['segment'] = df_scored.apply(assign_segment, axis=1)

seg = df_scored.groupby('segment').agg(
    customers        =('CLV','count'),
    avg_churn_proba  =('churn_proba','mean'),
    avg_CLV          =('CLV','mean'),
    total_rev_at_risk=('revenue_at_risk','sum'),
).round(2).sort_values('total_rev_at_risk', ascending=False)
print(seg)

# 2x2 heatmap of revenue-at-risk concentration
df_scored['value_tier'] = np.where(df_scored['CLV']>=value_threshold,'High Value','Low Value')
df_scored['risk_tier']  = np.where(df_scored['churn_proba']>=RISK_THRESHOLD,'High Risk','Low Risk')
pivot = df_scored.pivot_table(index='value_tier', columns='risk_tier',
                              values='revenue_at_risk', aggfunc='sum')
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='Reds')
plt.title('Revenue-at-Risk by Segment ($)'); plt.tight_layout(); plt.show()


In [ ]:
intervention = {
    'Priority Retention':    {'offer_cost': 80, 'success_rate': 0.35, 'action': 'Premium save offer + device upgrade'},
    'Standard Intervention': {'offer_cost': 30, 'success_rate': 0.25, 'action': 'Automated discount / plan re-fit'},
    'Protect & Grow':        {'offer_cost': 15, 'success_rate': 0.10, 'action': 'Loyalty nudge / upsell'},
    'Monitor':               {'offer_cost': 0,  'success_rate': 0.00, 'action': 'No spend — monitor only'},
}

rows = []
for s, p in intervention.items():
    g = df_scored[df_scored['segment'] == s]
    n = len(g)
    campaign_cost = n * p['offer_cost']
    revenue_saved = (g['churn_proba'] * p['success_rate'] * g['CLV']).sum()
    net_impact    = revenue_saved - campaign_cost
    roi           = net_impact / campaign_cost if campaign_cost else np.nan
    rows.append({'segment': s, 'customers': n, 'action': p['action'],
                 'campaign_cost': round(campaign_cost),
                 'revenue_saved': round(revenue_saved),
                 'net_impact': round(net_impact),
                 'ROI_x': round(roi, 1) if campaign_cost else None})

roi_table = pd.DataFrame(rows).set_index('segment')
print(roi_table.to_string())
print(f"\nTotal net impact (all interventions): ${roi_table['net_impact'].sum():,.0f}")
roi_table.to_csv('revenue_at_risk_roi.csv')


In [ ]:
from sklearn.cluster import KMeans

priority = df_scored[df_scored['segment'] == 'Priority Retention'].copy()
why_cols = ['eqpdays','custcare_Mean','drop_vce_Mean','blck_vce_Mean','change_mou','ovrrev_Mean']

why_df = priority[why_cols].fillna(priority[why_cols].median())
why_scaled = StandardScaler().fit_transform(why_df)
priority['exit_reason'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(why_scaled)

exit_summary = priority.groupby('exit_reason').agg(
    customer_count       =('rev_Mean','count'),
    avg_monthly_revenue  =('rev_Mean','mean'),
    total_revenue_at_risk=('revenue_at_risk','sum'),
    avg_equipment_age    =('eqpdays','mean'),
    avg_support_calls    =('custcare_Mean','mean'),
    avg_dropped_calls    =('drop_vce_Mean','mean'),
    avg_overage_charges  =('ovrrev_Mean','mean'),
).round(2)
print(exit_summary)
exit_summary.to_csv('high_value_exit_report.csv')


In [ ]:
priority_row = roi_table.loc['Priority Retention']
print("PROPOSAL HEADLINE")
print("-" * 60)
print(f"Targeting the {int(priority_row['customers']):,} 'Priority Retention' customers")
print(f"(high-value + high-risk) with a {intervention['Priority Retention']['action']}")
print(f"costs ${priority_row['campaign_cost']:,.0f} and is expected to retain")
print(f"${priority_row['revenue_saved']:,.0f} in customer lifetime value —")
print(f"a net gain of ${priority_row['net_impact']:,.0f} ({priority_row['ROI_x']}x ROI).")
